# Question 1 


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

pio.renderers.default = 'notebook'

In [ ]:
df = pd.read_csv('Student_Performance.csv')
print(df.shape)
df.head()

In [ ]:
# Encode categorical column
df['Extracurricular Activities'] = df['Extracurricular Activities'].map({'Yes': 1, 'No': 0})

feature_cols = [
    'Hours Studied',
    'Previous Scores',
    'Extracurricular Activities',
    'Sleep Hours',
    'Sample Question Papers Practiced'
]
target_col = 'Performance Index'

X = df[feature_cols].values.astype(float)
y = df[target_col].values.astype(float)

# Compute mean and std on the full feature matrix (will re-apply after splitting)
# Split first, then standardise using train statistics only
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(X_temp_raw, y_temp, test_size=0.5, random_state=42)

# Z-score standardisation using train statistics
mu    = X_train_raw.mean(axis=0)
sigma = X_train_raw.std(axis=0)

X_train = (X_train_raw - mu) / sigma
X_val   = (X_val_raw   - mu) / sigma
X_test  = (X_test_raw  - mu) / sigma

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

$$\hat{\boldsymbol{\beta}} = (X^\top X)^{-1} X^\top \mathbf{y}$$
where $X$ has a prepended column of ones for the bias.

In [ ]:
# Prepend bias column
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

Xb_train = add_bias(X_train)
Xb_test  = add_bias(X_test)

# Normal equation
beta = np.linalg.pinv(Xb_train.T @ Xb_train) @ Xb_train.T @ y_train

# Coefficients table
coeff_df = pd.DataFrame({'Feature': ['Bias'] + feature_cols, 'Coefficient': beta})
print(coeff_df.to_string(index=False))

In [ ]:
y_pred_scratch = Xb_test @ beta

mse_scratch = mean_squared_error(y_test, y_pred_scratch)
r2_scratch  = r2_score(y_test, y_pred_scratch)

print(f"Model 1 (From Scratch) — MSE: {mse_scratch:.4f}, R²: {r2_scratch:.4f}")

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_sklearn = lr.predict(X_test)

mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
r2_sklearn  = r2_score(y_test, y_pred_sklearn)

print(f"Model 2 (Scikit-learn) — MSE: {mse_sklearn:.4f}, R²: {r2_sklearn:.4f}")

In [ ]:
summary = pd.DataFrame({
    'Model': ['From Scratch (Normal Eq.)', 'Scikit-learn'],
    'MSE':   [mse_scratch, mse_sklearn],
    'R²':    [r2_scratch,  r2_sklearn]
})
print(summary.to_string(index=False))

In [ ]:
# Sort by actual for cleaner line plot
idx = np.argsort(y_test)
y_sorted      = y_test[idx]
pred_scratch  = y_pred_scratch[idx]
pred_sklearn  = y_pred_sklearn[idx]

# Sample 500 points for readability
sample = np.linspace(0, len(idx)-1, 500, dtype=int)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sample, y=y_sorted[sample],
    mode='lines', name='Actual',
    line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=sample, y=pred_scratch[sample],
    mode='lines', name='From Scratch',
    line=dict(color='tomato', width=1.5, dash='dash')
))
fig.add_trace(go.Scatter(
    x=sample, y=pred_sklearn[sample],
    mode='lines', name='Scikit-learn',
    line=dict(color='seagreen', width=1.5, dash='dot')
))

fig.update_layout(
    title='Actual vs Predicted Performance Index (Test Set)',
    xaxis_title='Sorted Sample Index',
    yaxis_title='Performance Index',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99)
)
fig.show()

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=y_pred_scratch, y=y_test - y_pred_scratch,
    mode='markers', marker=dict(color='slateblue', opacity=0.4, size=4),
    name='Residuals'
))
fig2.add_hline(y=0, line_color='red', line_dash='dash')
fig2.update_layout(
    title='Residuals vs Fitted (From-Scratch Model)',
    xaxis_title='Fitted Values',
    yaxis_title='Residuals',
    template='plotly_white'
)
fig2.show()